# Cuaderno 1: Limpieza y Transformacion de Datos (ETL)
**Proyecto Final — Analisis de Subsidios de Vivienda en Colombia**

---
Este cuaderno documenta el proceso ETL completo: carga de datos crudos, inspeccion, limpieza, transformacion y exportacion del dataset integrado listo para analisis.

## 1. Importacion de Librerias

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:,.2f}'.format)
plt.rcParams['figure.figsize'] = (12, 5)
print("Librerias cargadas correctamente")

## 2. Carga de Datasets Originales

In [ ]:
# Dataset 1: Deficit y Mejoramiento de Vivienda
df1_raw = pd.read_csv(
    '../2. Dataset y Diccionario de Datos/Datasets Originales/Dataset 1 Déficit y Mejoramiento de Vivienda.csv',
    encoding='utf-8', low_memory=False
)

# Dataset 2: Subsidios de Vivienda (Interes Prioritario)
df2_raw = pd.read_csv(
    '../2. Dataset y Diccionario de Datos/Datasets Originales/Dataset 2 Subsidios de Vivienda (Interés Prioritario).csv',
    encoding='utf-8', low_memory=False
)

# Dataset 3: Subsidios en Modalidad de Arrendamiento
df3_raw = pd.read_csv(
    '../2. Dataset y Diccionario de Datos/Datasets Originales/Dataset 3 Subsidios en Modalidad de Arrendamiento.csv',
    encoding='utf-8', low_memory=False
)

print(f"Dataset 1 — Filas: {df1_raw.shape[0]:,}, Columnas: {df1_raw.shape[1]}")
print(f"Dataset 2 — Filas: {df2_raw.shape[0]:,}, Columnas: {df2_raw.shape[1]}")
print(f"Dataset 3 — Filas: {df3_raw.shape[0]:,}, Columnas: {df3_raw.shape[1]}")

## 3. Inspeccion Inicial

In [ ]:
df1_raw.head(3)

In [ ]:
df2_raw.head(3)

In [ ]:
df3_raw.head(3)

In [ ]:
# Tipos de datos y nulos por dataset
for nombre, df in [("Dataset 1", df1_raw), ("Dataset 2", df2_raw), ("Dataset 3", df3_raw)]:
    print(f"\n{'='*50}\n  {nombre}\n{'='*50}")
    info = pd.DataFrame({
        'Tipo': df.dtypes,
        'Nulos': df.isnull().sum(),
        '% Nulos': (df.isnull().mean() * 100).round(2)
    })
    nulos = info[info['Nulos'] > 0]
    if len(nulos):
        print(nulos.to_string())
    else:
        print("  Sin valores nulos")

## 4. Limpieza — Dataset 1 (Deficit y Mejoramiento de Vivienda)

In [ ]:
df1 = df1_raw.copy()

# Normalizar nombres de columnas
df1.columns = df1.columns.str.strip().str.lower().str.replace('"', '')

# Renombrar columnas clave
rename_map1 = {
    'tipoacuerdo': 'tipo_acuerdo', 'municipio': 'municipio', 'cod_municipio': 'cod_municipio',
    'subregion': 'subregion', 'fase': 'fase', 'nropobla': 'nro_pobla',
    'aportegob': 'aporte_gob', 'aportemun': 'aporte_mun', 'aporteotr': 'aporte_otro',
    'fechainicio': 'fecha_inicio', 'fechafin': 'fecha_fin', 'cantidadtotal': 'cantidad_total',
}
df1.rename(columns={k: v for k, v in rename_map1.items() if k in df1.columns}, inplace=True)

# Convertir fechas
for col in ['fecha_inicio', 'fecha_fin']:
    if col in df1.columns:
        df1[col] = pd.to_datetime(df1[col], errors='coerce')

# Eliminar duplicados
antes = len(df1)
df1.drop_duplicates(inplace=True)
print(f"Duplicados eliminados: {antes - len(df1)}")

# Eliminar columnas con >70% nulos
cols_altas_nulos = df1.columns[df1.isnull().mean() > 0.70].tolist()
df1.drop(columns=cols_altas_nulos, inplace=True, errors='ignore')
print(f"Columnas eliminadas por >70% nulos: {cols_altas_nulos}")

# Limpiar aportes monetarios
for col in ['aporte_gob', 'aporte_mun', 'aporte_otro']:
    if col in df1.columns:
        df1[col] = pd.to_numeric(df1[col], errors='coerce').fillna(0)

print(f"\nDataset 1 limpio — Shape: {df1.shape}")
df1.head(3)

## 5. Limpieza — Dataset 2 (Subsidios Interes Prioritario)

In [ ]:
df2 = df2_raw.copy()
df2.columns = (df2.columns.str.strip().str.lower()
               .str.replace('[^a-z0-9_]', '_', regex=True)
               .str.replace('_+', '_', regex=True).str.strip('_'))

# Renombrar columna año
for col in ['a_o', 'anno', 'a__o']:
    if col in df2.columns:
        df2.rename(columns={col: 'anio'}, inplace=True)
        break

# Convertir fechas
for col in df2.select_dtypes(include='object').columns:
    if 'fecha' in col:
        df2[col] = pd.to_datetime(df2[col], errors='coerce')

antes = len(df2)
df2.drop_duplicates(inplace=True)
print(f"Duplicados eliminados: {antes - len(df2)}")
print(f"Dataset 2 limpio — Shape: {df2.shape}")
df2.head(3)

## 6. Limpieza — Dataset 3 (Subsidios Arrendamiento)

In [ ]:
df3 = df3_raw.copy()
df3.columns = (df3.columns.str.strip().str.lower()
               .str.replace('[^a-z0-9_]', '_', regex=True)
               .str.replace('_+', '_', regex=True).str.strip('_'))

# Unificar nombres de columnas
rename_map3 = {}
for col in df3.columns:
    if 'divipola' in col and 'departamento' in col:
        rename_map3[col] = 'cod_divipola_departamento'
    elif 'divipola' in col and 'municipio' in col:
        rename_map3[col] = 'cod_divipola_municipio'
    elif 'asignaci' in col or col.startswith('a_o') or col.startswith('a__o'):
        rename_map3[col] = 'anio_de_asignacion'
df3.rename(columns=rename_map3, inplace=True)

if 'valor_asignado' in df3.columns:
    df3['valor_asignado'] = pd.to_numeric(df3['valor_asignado'], errors='coerce')

antes = len(df3)
df3.drop_duplicates(inplace=True)
print(f"Duplicados eliminados: {antes - len(df3)}")
print(f"Dataset 3 limpio — Shape: {df3.shape}")
df3.head(3)

## 7. Carga de Datos del Modelo Estrella (KNIME)

In [ ]:
hechos = pd.read_csv('../2. Dataset y Diccionario de Datos/CSV Final Knime/Hechos_Final.csv', encoding='utf-8-sig')
cat_municipio = pd.read_csv('../2. Dataset y Diccionario de Datos/CSV Final Knime/CatMunicipio_clean.csv', encoding='utf-8-sig')
cat_programa = pd.read_csv('../2. Dataset y Diccionario de Datos/CSV Final Knime/CatPrograma_clean.csv', encoding='utf-8-sig')
cat_tiempo = pd.read_csv('../2. Dataset y Diccionario de Datos/CSV Final Knime/CatTiempo_clean.csv', encoding='utf-8-sig')

print(f"Hechos_Final  : {hechos.shape}")
print(f"CatMunicipio  : {cat_municipio.shape}")
print(f"CatPrograma   : {cat_programa.shape}")
print(f"CatTiempo     : {cat_tiempo.shape}")
hechos.head()

## 8. Validacion de Integridad del Modelo Estrella

In [ ]:
# Municipio
mun_hechos = set(hechos['cod_divipola_mun'].dropna().astype(int))
mun_dim = set(cat_municipio['cod_divipola_mun'].dropna().astype(int))
print(f"Municipios sin dimension: {len(mun_hechos - mun_dim)}")

# Programa
prog_hechos = set(hechos['programa'].dropna().str.strip().str.upper())
prog_dim = set(cat_programa['programa'].dropna().str.strip().str.upper())
orphans = prog_hechos - prog_dim
print(f"Programas sin dimension: {len(orphans)}")
if orphans:
    print("  ->", orphans)

print("\nValidacion completada")

## 9. Resumen del Proceso ETL

In [ ]:
resumen = pd.DataFrame({
    'Dataset': ['DS1 Original', 'DS1 Limpio', 'DS2 Original', 'DS2 Limpio',
                'DS3 Original', 'DS3 Limpio', 'Hechos Final (KNIME)'],
    'Filas': [df1_raw.shape[0], df1.shape[0], df2_raw.shape[0], df2.shape[0],
              df3_raw.shape[0], df3.shape[0], hechos.shape[0]],
    'Columnas': [df1_raw.shape[1], df1.shape[1], df2_raw.shape[1], df2.shape[1],
                 df3_raw.shape[1], df3.shape[1], hechos.shape[1]]
})

fig, ax = plt.subplots(figsize=(10, 4))
ax.axis('off')
tabla = ax.table(cellText=resumen.values, colLabels=resumen.columns,
                 loc='center', cellLoc='center')
tabla.auto_set_font_size(False)
tabla.set_fontsize(10)
tabla.scale(1.2, 1.8)
plt.title("Resumen ETL — Datasets del Proyecto", fontsize=13, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

print(resumen.to_string(index=False))
print("\nProceso ETL finalizado correctamente.")